# Week 4: Combining and Reshaping Global Data

**Learning objective:** Merge related tables safely and reshape data between long and wide forms.

**Expected outputs:** key checks, a successful merge, an unmatched-row audit, and long/wide tables.

**Data source:** Synthetic course data. [Open in Colab](https://colab.research.google.com/github/GSIS-DS/data-science-ai-global-decision-making/blob/main/notebooks/week-04/04_combining_reshaping_global_data.ipynb).

In [ ]:
from pathlib import Path
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/GSIS-DS/data-science-ai-global-decision-making/main/data/sample/global_indicators_sample.csv"
LOCAL_PATHS = [
    Path("data/sample/global_indicators_sample.csv"),
    Path("../../data/sample/global_indicators_sample.csv"),
]
local_path = next((path for path in LOCAL_PATHS if path.exists()), None)
data_source = str(local_path) if local_path else DATA_URL
df = pd.read_csv(data_source)
print(f"Loaded {len(df)} rows from {data_source}")
df.head()

## 1. Define tables, grain, and keys

Both tables should contain one row per ISO3 code and year. Check that claim before merging.

In [ ]:
macro = df[['country', 'iso3', 'year', 'gdp_growth', 'inflation', 'region']].copy()
trade = df[['iso3', 'year', 'exports_usd_bn', 'imports_usd_bn', 'trade_openess_percent']].copy()
keys = ['iso3', 'year']
print('Macro duplicate keys:', macro.duplicated(keys).sum())
print('Trade duplicate keys:', trade.duplicated(keys).sum())

In [ ]:
merged = macro.merge(trade, on=keys, how='left', validate='one_to_one', indicator=True)
display(merged['_merge'].value_counts())
assert (merged['_merge'] == 'both').all()
merged.head()

## 2. Deliberate unmatched-row example

Real sources rarely align perfectly. Remove one trade row to simulate a coverage gap, then audit rather than silently dropping the country-year.

In [ ]:
trade_incomplete = trade.iloc[1:].copy()
audit_merge = macro.merge(trade_incomplete, on=keys, how='left', validate='one_to_one', indicator=True)
unmatched = audit_merge.loc[audit_merge['_merge'] != 'both', keys + ['country', '_merge']]
unmatched

A left join preserves every macro row. The unmatched record now has missing trade measures. Before analysis, investigate source coverage and decide whether exclusion, imputation, or an explicit limitation is justified.

## 3. Reshape between long and wide

In [ ]:
long_df = (merged[['iso3', 'year', 'gdp_growth', 'inflation']]
           .melt(id_vars=['iso3', 'year'], var_name='metric', value_name='value'))
long_df.head(8)

In [ ]:
wide_df = (long_df.pivot(index=['iso3', 'year'], columns='metric', values='value')
           .reset_index()
           .rename_axis(columns=None))
wide_df.head()

## Exercises and source-documentation checkpoint

1. Change the successful merge to an inner join. Report whether the row count changes. **Expected output:** before/after counts plus one sentence.
2. Create a wide table with years as columns for one metric. **Expected output:** one row per ISO3 and one column per year.
3. Record source file, table grain, keys, join type, validation rule, unmatched-row count, and one merge risk.

Never describe a merge as successful only because Python produced a table.